# 15 Surveys and designs: the api samples of Lumley (2010), design tables and randomisation lists

- `table.use_survey_design` and the words that then answer for the population (`find_average_of`,
  `find_total_of`, `find_median_of`, `find_percentile_of`, `find_percent_of_rows_where`, `count_rows`,
  `explain`, `compare_averages_of`, `compare_counts_of`) on the **California API samples** of the R survey
  package: the stratified sample `apistrat` (200 schools in 3 strata) and the one-stage cluster sample
  `apiclus1` (183 schools in 15 districts). Lumley (2010, *Complex Surveys*, chapter 2) printed the
  stratified mean API 2000 662.29 (standard error 9.4089), the total enrolment 3,687,178 (114,642) and the
  cluster mean 644.17 (23.542).
- `sdata.make_design_table`, `sdata.make_random_assignments` and `table.keep_random_rows`, where a
  deterministic reference exists: the same R package with the same seed and random number generator
  (agricolae with `kinds = "Mersenne-Twister"`, blockrand after `set.seed(..., kind = "Mersenne-Twister",
  sample.kind = "Rejection")`), or pandas `groupby().sample()` with the same `random_state`; and the
  published properties of each design (every combination once in each block; each level once in every row
  and column of a latin square; Yates' standard order, Montgomery 2017; permuted blocks, Schulz & Grimes
  2002).

**Only simplydata runs in this notebook.** The reference values (`references/15_surveys_and_designs.csv`, from
`scripts/references/15_surveys_and_designs.py`) are listed below; every check prints its source.

References and tolerances:
- **Printed values** (Lumley 2010): half a unit of the last printed digit.
- **NumPy, textbook formulas** (Lohr 2010, *Sampling: Design and Analysis*, 2nd ed.: the stratified
  estimator with its finite population correction, section 3.2, and the linearised variance of the weighted
  total and of the ratio mean, section 9.1, written out in NumPy/pandas with the sampling weights `pw`): a relative difference of 1e-10 (closed formulas).
- **R survey** (`svydesign`, `svymean`, `svytotal`, `svyquantile`, `svyciprop`, `svyby`, `svyglm`,
  `svyttest`, `svychisq`), run here with the same design: a relative difference of 1e-9 (the estimates
  are weighted least squares or estimating equations, the level the README uses for least squares).
- **Designs and random rows**: exact. A layout (each column of the plots in turn, a level given as text
  counted by its place in the list of levels) must be the same as the reference layout, cell by cell; the kept
  schools must be the same school numbers. The published properties of each design are counted on simplydata's
  own table with simplydata words (`count_rows`, `count_different_values_in`), and each count must be 0.


In [1]:
import simplydata as sdata

from checks import ReferenceChecks, quiet, read_numbers, read_row, read_rows, versions

NOTEBOOK = "15_surveys_and_designs"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The stratified sample: `use_survey_design(weights=, strata=, population_sizes=)`

`pw` is each school's sampling weight, `stype` the stratum (elementary, middle, high) and `fpc` the number
of schools of that type in California. One full printed answer first.

In [2]:
strat = quiet(sdata.table, "../data/fetched/apistrat.csv", separator=",")
strat.use_survey_design(weights="pw", strata="stype", population_sizes="fpc")
strat.find_average_of("api00")

This table is now a survey (weights "pw", strata "stype", population sizes "fpc"; 200 rows, 3 strata, 197 degrees of freedom of the design, R survey::svydesign()).
From now on find_average_of, find_total_of, find_median_of, find_percentile_of, find_percent_of_rows_where, count_rows, explain, compare_averages_of and compare_counts_of describe the population, from the survey design; other statistics words refuse. Turn it off with table.use_survey_design().
"fpc" is read as the number of rows in the population of each stratum, as survey::svydesign(fpc =) reads it; this makes the standard errors smaller when much of a stratum was sampled.
As a survey (weights "pw", strata "stype", population sizes "fpc"): numbers describe the population.
The population average of "api00": 662.29 (95 % CI 643.73 to 680.84).
Method: R survey::svymean; t interval on the design's 197 degrees of freedom (survey's own confint() uses the normal curve).


,average of api00,standard error,average of api00 95 % CI low,average of api00 95 % CI high,degrees of freedom
1,662.287363,9.408941,643.732188,680.842538,197.0


In [3]:
DS = "api stratified sample"
mean = read_row(quiet(strat.find_average_of, "api00"))
total = read_row(quiet(strat.find_total_of, "enroll"))
checks.check_values(DS, "after use_survey_design (strata), find_average_of / find_total_of",
                    {"mean of api00": mean["average of api00"], "standard error of the mean": mean["standard error"],
                     "95 % CI low of the mean": mean["average of api00 95 % CI low"],
                     "total of enroll": total["total of enroll"], "standard error of the total": total["standard error"],
                     "95 % CI high of the total": total["total of enroll 95 % CI high"],
                     "degrees of freedom": mean["degrees of freedom"]})

agrees  api stratified sample | after use_survey_design (strata), find_average_of / find_total_of | mean of api00: simplydata 662.287363159, Lumley (2010), printed 662.29; difference 2.64e-03 (allowed 5e-03)
      Source: Lumley (2010), printed (printed) - Lumley T (2010). Complex Surveys: A Guide to Analysis Using R. Wiley, chapter 2
agrees  api stratified sample | after use_survey_design (strata), find_average_of / find_total_of | mean of api00: simplydata 662.287363159, NumPy, textbook formula (Lohr 2010) 662.287363159; relative difference 3.43e-16 (allowed 1e-10)
      Source: NumPy, textbook formula (Lohr 2010) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/15_surveys_and_designs.py
agrees  api stratified sample | after use_survey_design (strata), find_average_of / find_total_of | mean of api00: simplydata 662.287363159, R survey svymean / svytotal 662.287363159; relative difference 0.00e+00 (allowed 1e-09)
      Source: R survey s

16

### Median, percentile, percent of rows, counts, a model, two groups and a table of counts

Each against R survey with the same design: `svyquantile(qrule = "math", ci = TRUE)` (Woodruff's interval),
`svyciprop(method = "logit")`, `svyby(svytotal)` and `svymean` of the factor, `svyglm`, `svyttest` and
`svychisq` (Rao-Scott F).

In [4]:
median = read_row(quiet(strat.find_median_of, "api00"))
p90 = read_row(quiet(strat.find_percentile_of, "api00", 90))
percent = read_row(quiet(strat.find_percent_of_rows_where, "sch.wide", is_equal_to="Yes"))
counts = {row["awards"]: row for row in rows(strat.count_rows, for_each_group_in="awards")}
shares = {row["awards"]: row for row in rows(strat.count_rows, for_each_group_in="awards", in_percent=True)}
model = {row["term"]: row for row in rows(strat.explain, "api00", using=["ell", "meals"])}
two = read_row(quiet(strat.compare_averages_of, "api00", for_each_group_in="awards"))
chi = read_row(quiet(strat.compare_counts_of, "awards", for_each_group_in="stype"))
checks.check_values(DS, "after use_survey_design (strata), other words", {
    "median": median["median of api00"], "median 95 % CI low": median["median of api00 95 % CI low"],
    "median 95 % CI high": median["median of api00 95 % CI high"], "median standard error": median["standard error"],
    "percentile 90": p90["percentile 90 of api00"], "percentile 90 95 % CI low": p90["percentile 90 of api00 95 % CI low"],
    "percent sch.wide Yes": percent["percent"], "percent 95 % CI low": percent["percent 95 % CI low"],
    "percent 95 % CI high": percent["percent 95 % CI high"],
    "schools with an award": counts["Yes"]["population count"],
    "schools with an award, standard error": counts["Yes"]["standard error"],
    "percent with an award": shares["Yes"]["percent"],
    "intercept": model["(Intercept)"]["estimate"], "ell": model["ell"]["estimate"], "meals": model["meals"]["estimate"],
    "meals standard error": model["meals"]["standard error"], "ell p-value": model["ell"]["p-value"],
    "difference (awards)": two["difference"], "t (awards)": two["t"], "p-value (awards)": two["p-value"],
    "Rao-Scott F": chi["F"], "Rao-Scott p-value": chi["p-value"],
    "Rao-Scott design degrees of freedom": chi["design degrees of freedom"]})

agrees  api stratified sample | after use_survey_design (strata), other words | median: simplydata 668, R survey 668; relative difference 0.00e+00 (allowed 1e-09)
      Source: R survey (run reference) - https://cran.r-project.org/package=survey (version 4.5)
agrees  api stratified sample | after use_survey_design (strata), other words | median 95 % CI low: simplydata 642, R survey 642; relative difference 0.00e+00 (allowed 1e-09)
      Source: R survey (run reference) - https://cran.r-project.org/package=survey (version 4.5)
agrees  api stratified sample | after use_survey_design (strata), other words | median 95 % CI high: simplydata 694, R survey 694; relative difference 0.00e+00 (allowed 1e-09)
      Source: R survey (run reference) - https://cran.r-project.org/package=survey (version 4.5)
agrees  api stratified sample | after use_survey_design (strata), other words | median standard error: simplydata 13.1840557882, R survey 13.1840557882; relative difference 0.00e+00 (allowed 1e-0

23

## The one-stage cluster sample: `use_survey_design(weights=, clusters=, population_sizes=)`

15 of California's 757 school districts sampled at random, every school in them kept. The textbook
reference is the ratio mean with its linearised variance (Lohr 2010, section 9.1), written out in the
reference script with each district as one sampling unit.

In [5]:
clus = quiet(sdata.table, "../data/fetched/apiclus1.csv", separator=",")
quiet(clus.use_survey_design, weights="pw", clusters="dnum", population_sizes="fpc")
found = read_row(quiet(clus.find_average_of, "api00"))
checks.check_values("api cluster sample", "after use_survey_design (clusters), find_average_of",
                    {"mean of api00": found["average of api00"], "standard error of the mean": found["standard error"],
                     "degrees of freedom": found["degrees of freedom"]})

agrees  api cluster sample | after use_survey_design (clusters), find_average_of | mean of api00: simplydata 644.169398907, Lumley (2010), printed 644.17; difference 6.01e-04 (allowed 5e-03)
      Source: Lumley (2010), printed (printed) - Lumley T (2010). Complex Surveys: A Guide to Analysis Using R. Wiley, chapter 2
agrees  api cluster sample | after use_survey_design (clusters), find_average_of | mean of api00: simplydata 644.169398907, NumPy, textbook formula (Lohr 2010) 644.169398907; relative difference 0.00e+00 (allowed 1e-10)
      Source: NumPy, textbook formula (Lohr 2010) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/15_surveys_and_designs.py
agrees  api cluster sample | after use_survey_design (clusters), find_average_of | mean of api00: simplydata 644.169398907, R survey svymean 644.169398907; relative difference 0.00e+00 (allowed 1e-09)
      Source: R survey svymean (run reference) - https://search.r-project.org/CRAN/ref

8

## Design tables: `sdata.make_design_table` against R agricolae with the same seed

simplydata says it runs agricolae with R's Mersenne-Twister generator (`kinds = "Mersenne-Twister"`), so the
same seed must give the same layout, plot by plot. agricolae numbers the levels 1, 2, ... in `design.ab`;
they are mapped back to the level names. The published properties are checked on simplydata's table.

In [6]:
sdata.make_design_table({"a": [0, 1], "b": [0, 1], "c": [0, 1]}, repeats=2, with_seed=1)

A design table of 16 plots: the 2 x 2 x 2 = 8 combinations of a, b, c; every plot in one random order, each combination 2 times (R agricolae::design.ab(design = "crd")).
Work through the plots in the order of "plot"; the randomisation spreads unknown differences (soil, position, time of day) evenly over the treatments.
"standard order" numbers the combinations as R expand.grid() lists them (the first factor changes fastest), the order of design textbooks.
with_seed=1 makes the same table again (R's Mersenne-Twister generator, as after set.seed() in R 4.6.2; agricolae's own default generator, Super-Duper, gives another layout).


,plot,repeat,a,b,c,standard order
1,1,1,1,0,0,2
2,2,1,0,0,1,5
3,3,1,0,1,1,7
4,4,1,0,0,0,1
5,5,2,0,0,0,1
6,6,1,1,1,0,4
7,7,1,1,0,1,6
8,8,2,0,0,1,5
9,9,2,1,1,0,4
10,10,1,0,1,0,3


In [7]:
def layout(table, levels):
    """The compared columns of a layout as one list of numbers, column after column; a level given as text counts
    as its place in the list of levels (counting from 1)."""
    plots = read_rows(table)
    numbers = []
    for column, column_levels in levels.items():
        numbers += [row[column] if column_levels is None else column_levels.index(row[column]) + 1 for row in plots]
    return numbers


def how_many_rows(found, column, wrong):
    return sum(1 for row in found if wrong(row[column]))


DESIGNS = [  # (title, simplydata call, the columns compared with agricolae and their levels)
    ("2 x 2 x 2 factorial, completely random, seed 1",
     dict(factors={"a": [0, 1], "b": [0, 1], "c": [0, 1]}, repeats=2, with_seed=1), {"a": None, "b": None, "c": None}),
    ("2 x 2 factorial in 4 randomised blocks, seed 3",
     dict(factors={"a": [0, 1], "b": [0, 1]}, repeats=4, of_kind="randomised blocks", with_seed=3),
     {"block": None, "a": None, "b": None}),
    ("one factor (3 varieties) in 4 randomised blocks, seed 5",
     dict(factors={"variety": ["A", "B", "C"]}, repeats=4, of_kind="randomised blocks", with_seed=5),
     {"block": None, "variety": ["A", "B", "C"]}),
    ("one factor (4 levels), completely random, 3 repeats, seed 6",
     dict(factors={"v": ["p", "q", "r", "s"]}, repeats=3, with_seed=6), {"v": ["p", "q", "r", "s"]}),
    ("latin square of 4 levels, seed 2",
     dict(factors={"v": ["A", "B", "C", "D"]}, of_kind="latin square", with_seed=2),
     {"row": None, "column": None, "v": ["A", "B", "C", "D"]}),
    ("split plot, irrigation on whole plots, 3 blocks, seed 4",
     dict(factors={"irrigation": ["dry", "wet"], "variety": ["A", "B", "C"]}, repeats=3, of_kind="split plot",
          whole_plots="irrigation", with_seed=4),
     {"block": None, "irrigation": ["dry", "wet"], "variety": ["A", "B", "C"]}),
]
for title, call, levels in DESIGNS:
    plots = quiet(sdata.make_design_table, **call)
    M = "make_design_table"
    checks.check(title, M, "agricolae's layout, cell by cell", layout(plots, levels))
    factors = list(call["factors"])
    kind = call.get("of_kind", "completely random")
    if kind == "latin square":
        k = len(call["factors"]["v"])
        for line in ("row", "column"):
            found = rows(plots.count_different_values_in, "v", for_each_group_in=line)
            checks.check(title, M, f"{line}s missing a level", how_many_rows(found, "different values", lambda n: n != k))
    elif kind == "split plot":
        found = rows(plots.count_different_values_in, "irrigation", for_each_group_in="whole plot")
        checks.check(title, M, "whole plots with more than one irrigation",
                     how_many_rows(found, "different values", lambda n: n != 1))
        found = rows(plots.count_different_values_in, "variety", for_each_group_in="whole plot")
        checks.check(title, M, "whole plots missing a variety", how_many_rows(found, "different values", lambda n: n != 3))
    else:
        combinations = 1
        for values in call["factors"].values():
            combinations *= len(values)
        if kind == "randomised blocks":  # every combination exactly once in every block
            found = rows(plots.count_rows, for_each_group_in=["block"] + factors)
            wrong = how_many_rows(found, "count", lambda n: n != 1) + combinations * call["repeats"] - len(found)
        else:  # every combination exactly `repeats` times
            found = rows(plots.count_rows, for_each_group_in=factors)
            wrong = how_many_rows(found, "count", lambda n: n != call["repeats"]) + combinations - len(found)
        checks.check(title, M, "combinations not used as often as the design says", wrong)
        if len(factors) > 1:
            # Yates' standard order (Montgomery 2017): (1), a, b, ab, c, ... - the first factor changes fastest
            wrong = 0
            for row in read_rows(plots):
                expected, step = 1, 1
                for f in factors:
                    expected += call["factors"][f].index(row[f]) * step
                    step *= len(call["factors"][f])
                wrong += row["standard order"] != expected
            checks.check(title, M, "plots with a wrong standard order", wrong)

agrees  2 x 2 x 2 factorial, completely random, seed 1 | make_design_table | agricolae's layout, cell by cell: largest relative difference over 48 values: simplydata 0, R agricolae (same seed, Mersenne-Twister) 0; difference 0.00e+00 (allowed 0e+00)
      Source: R agricolae (same seed, Mersenne-Twister) (run reference) - https://cran.r-project.org/package=agricolae (version 1.3.8)
agrees  2 x 2 x 2 factorial, completely random, seed 1 | make_design_table | combinations not used as often as the design says: simplydata 0, definition of the design 0; difference 0.00e+00 (allowed 0e+00)
      Source: definition of the design (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/15_surveys_and_designs.py
agrees  2 x 2 x 2 factorial, completely random, seed 1 | make_design_table | plots with a wrong standard order: simplydata 0, Yates' standard order (Montgomery 2017) 0; difference 0.00e+00 (allowed 0e+00)
      Source: Yates' standard order (Montg

## Randomisation lists: `sdata.make_random_assignments` against R blockrand with the same seed

Permuted blocks for two centres (Bergen, Oslo). The reference sets the seed as simplydata says it does
(`set.seed(seed, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")`) and calls
`blockrand()` once for each centre, in that order (blockrand counts block sizes in multiples of the number of
groups). Published property (Schulz & Grimes 2002): every block holds every group equally often.

In [8]:
for how_many, sizes, seed, groups in ((60, [4, 6], 1, ["A", "B"]), (45, [3, 6], 7, ["A", "B", "C"])):
    title = f"{how_many} places, {len(groups)} groups, blocks of {sizes}, two centres, seed {seed}"
    places = quiet(sdata.make_random_assignments, how_many=how_many, groups=groups, in_blocks_of=sizes,
                   for_each_of=["Bergen", "Oslo"], with_seed=seed)
    blocks = {}
    for row in rows(places.count_rows, for_each_group_in=["list", "block", "group"]):
        blocks.setdefault((row["list"], row["block"]), []).append(row["count"])
    unequal = sum(1 for counts in blocks.values() if len(counts) != len(groups) or len(set(counts)) != 1)
    checks.check(title, "make_random_assignments", "blocks where the groups are not equally often", unequal)
    checks.check(title, "make_random_assignments", "blockrand's list, cell by cell",
                 layout(places, {"list": ["Bergen", "Oslo"], "block": None, "block size": None, "group": groups}))

agrees  60 places, 2 groups, blocks of [4, 6], two centres, seed 1 | make_random_assignments | blocks where the groups are not equally often: simplydata 0, definition of permuted blocks 0; difference 0.00e+00 (allowed 0e+00)
      Source: definition of permuted blocks (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/15_surveys_and_designs.py
agrees  60 places, 2 groups, blocks of [4, 6], two centres, seed 1 | make_random_assignments | blockrand's list, cell by cell: largest relative difference over 488 values: simplydata 0, R blockrand (same seed) 0; difference 0.00e+00 (allowed 0e+00)
      Source: R blockrand (same seed) (run reference) - https://cran.r-project.org/package=blockrand (version 1.5)
agrees  45 places, 3 groups, blocks of [3, 6], two centres, seed 7 | make_random_assignments | blocks where the groups are not equally often: simplydata 0, definition of permuted blocks 0; difference 0.00e+00 (allowed 0e+00)
      Source: defin

## Random rows within groups: `table.keep_random_rows(for_each_group_in=)` against pandas

pandas `groupby().sample(n=..., random_state=...)` (or `frac=`) with the same seed; simplydata keeps the rows
in table order, so the kept school numbers are compared in order of their number.

In [9]:
for how_many, percent, seed in ((10, None, 2), (None, 10, 2), (5, None, 11)):
    schools = quiet(sdata.table, "../data/fetched/apistrat.csv", separator=",")
    if percent:
        quiet(schools.keep_random_rows, percent=percent, for_each_group_in="stype", with_seed=seed)
        title = f"api stratified sample, {percent} % of each school type, seed {seed}"
    else:
        quiet(schools.keep_random_rows, how_many, for_each_group_in="stype", with_seed=seed)
        title = f"api stratified sample, {how_many} of each school type, seed {seed}"
    kept = sorted(read_numbers(quiet(schools.get_column, "snum")))
    checks.check(title, "keep_random_rows", "rows kept", len(kept))
    checks.check(title, "keep_random_rows", "school numbers kept, in order", kept)

agrees  api stratified sample, 10 of each school type, seed 2 | keep_random_rows | rows kept: simplydata 30, pandas groupby().sample() 30; difference 0.00e+00 (allowed 0e+00)
      Source: pandas groupby().sample() (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=groupby
agrees  api stratified sample, 10 of each school type, seed 2 | keep_random_rows | school numbers kept, in order: largest relative difference over 30 values: simplydata 0, pandas groupby().sample() 0; difference 0.00e+00 (allowed 0e+00)
      Source: pandas groupby().sample() (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=groupby
agrees  api stratified sample, 10 % of each school type, seed 2 | keep_random_rows | rows kept: simplydata 20, pandas groupby().sample() 20; difference 0.00e+00 (allowed 0e+00)
      Source: pandas groupby().sample() (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=groupby
agrees  api str

## Result

In [10]:
results = checks.finish()

15_surveys_and_designs: 73 comparisons - 73 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
